# Image Preprocessing and OCR

This notebook takes raw scanned images of historical newspaper articles (`.png`, `.jpg`, `.jpeg` or `.pdf`) and applies a series of image enhancement steps to prepare them for optical character recognition (OCR). It then extracts machine-readable text from each image using [Tesseract](https://github.com/tesseract-ocr/tesseract), an open-source OCR engine.

This is the first step in the THISTLE pipeline. The OCR output saved at the end of this notebook is passed to `post_ocr_llm.ipynb` in the next stage.

**Run cells from top to bottom.** Each cell depends on variables created in earlier cells.




---



# Setting up

## Create a working directory and clone the repository.

You will only need to do this once. Before cloning, make sure that you have decided upon a working directory.

### **Locally**

If you are working locally, you can clone the repo into a location of your choice, e.g. your 'Documents' directory. The instructions below guide you through how to do this in your terminal.

First, change your working directory to 'Documents' (or the location of your choice):

```bash
cd ~/Documents
```

Next, create a new folder in 'Documents' with the title of your choice (e.g. 'THISTLE_repo'):

```bash
mkdir ~/THISTLE_repo  
```

Change your working directory to the project folder:

```bash
cd ~/THISTLE_repo
```

And print the path of your working directory to confirm you are in the right place:

```bash
pwd
```

Now, clone the repo into your working directory:

```bash
git clone https://github.com/jessicawitte92/THISTLE_project
```

Check this worked by printing the contents of your working directory:

```bash
ls
```

### **Google Colab**

You can clone the repo directly into this Colab notebook by using the command `!git clone https://github.com/jessicawitte92/THISTLE_project`. However, this will only clone the repo for the duration of this Colab session. To save files and outputs, you will need to first mount your Google Drive. The cells below will guide you through this process.

First, make sure you are connected to the T4 GPU. In the 'Runtime' menu, click on 'Change runtime type' and select 'T4 GPU' under 'Hardware accelerator'.

**NOTE:** you will need to be logged in with your Google account to connect your Drive and to use the GPU. Free access is limited; for larger projects, you may wish to consider high performance computing (HPC) infrastructure.

In [ ]:
# after runing this cell, you will need to approve the pop-up request to connect your Google Drive account.
from google.colab import drive
drive.mount('/content/drive')

After you have connected your Drive to this notebook, clone the repo into your project folder:

In [ ]:
# change the path below to your working directory in Google Drive before running this cell.
%cd "/content/drive/My Drive/your_project_folder"

In [ ]:
!git clone https://github.com/jessicawitte92/THISTLE_project.git

In [ ]:
# check that all contents of the repo cloned
# NOTE: update the path to match your working directory before running this cell.
%ls "/content/drive/My Drive/your_project_folder/THISTLE_project"



---



## Step 1: Install dependencies

Run this cell first to install all required Python packages.

- **In Google Colab:** packages are installed for your current session and will need reinstalling if you reconnect.
- **In Anaconda (local):** run this once per environment, or install via `requirements.txt` instead (see `code/README.md`).

> **System requirement:** You must also have **Tesseract OCR** installed on your machine before running this notebook. See `code/README.md` for platform-specific installation instructions. If you are using Google Colab, the cell below handles Tesseract installation automatically.

In [ ]:
# install required Python packages
!pip install pillow pdf2image pytesseract matplotlib

In [ ]:
# install Tesseract (Google Colab only — skip this cell or comment out (using a '#') if running locally)
!apt-get install -y tesseract-ocr

## Step 2: Import packages

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter
from pdf2image import convert_from_path
import pytesseract as pt

## Step 3: Set your image folder

Update `img_folder` below to point to the folder on your machine (or Google Drive) that contains your image files. The notebook accepts `.png`, `.jpg`, `.jpeg` and `.pdf` inputs.

**Note:** If you are using the THISTLE dataset, this folder is `data/imgs` inside the cloned repository.

In [ ]:
# paste the path to the folder containing your images
# example (Colab): "/content/drive/My Drive/your_project_folder/THISTLE_project/data/imgs"
# example (local): "/Users/yourname/Documents/THISTLE_project/data/imgs"
img_folder = "insert_your_path_to_image_folder"

# check the path is correct; if you do not receive a warning, proceed to the next code block
if not os.path.exists(img_folder):
    print(f'Warning: {img_folder} not found. Please update img_folder to your local path.')
else:
    print(f'Image folder found: {img_folder}')

## Step 4: Build the list of image files

This cell scans your image folder and collects all supported image file paths into a list. `.png`, `.jpg`, `.jpeg` and `.pdf` files are supported.

**Note for PDFs:** each page will be treated as a separate image in the next step.

In [ ]:
# collect paths to supported image files
img_files = []

for file in sorted(os.listdir(img_folder)):
    if file.endswith(('.png', '.jpg', '.jpeg', '.pdf')):
        file_path = os.path.join(img_folder, file)  # constructs the full path as a string
        img_files.append(file_path)

print(f'{len(img_files)} image file(s) found in {img_folder}')

## Step 5: Preprocess images and run OCR

This cell applies four preprocessing steps to each image to improve OCR accuracy:

1. **Greyscale conversion** — removes colour to ensure images are in greyscale
2. **Contrast enhancement** — sharpens the distinction between text and background
3. **Gaussian blur** — reduces noise and softens stray marks
4. **Binarisation** — converts the image to pure black and white using a pixel threshold

For each image, a side-by-side comparison of the original and preprocessed version is displayed. The preprocessed image is then passed to Tesseract for OCR and the result is stored in the `ocr_results` variable.

### Adjusting preprocessing parameters:
The parameters below were tuned for scanned images of *The Scotsman* from the THISTLE dataset. If you are working with a different dataset, you may need to experiment with them. For instance, you may wish to increase the `contrast_factor` or adjust `threshold` if the text in your dataset is blurry or faded; `blur_radius` can be adjusted to reduce background noise; if you have .pdf images, `dpi` adjusts resolution.


In [ ]:
# parameters — adjust for your dataset
contrast_factor = 3    # contrast enhancement (>1 increases contrast)
blur_radius = 0.8      # Gaussian blur radius for noise reduction
threshold = 142        # binarisation threshold at which pixels are converted to black or white (0-255)
dpi = 500              # resolution for PDF conversion (ignored for image files)

ocr_results = []

for idx, file_path in enumerate(img_files):
    filename = os.path.basename(file_path)

    # handle PDFs: convert each page to a PIL Image
    if file_path.endswith('.pdf'):
        pages = convert_from_path(file_path, dpi=dpi)
    else:
        pages = [Image.open(file_path)]

    for page_idx, orig_img in enumerate(pages):
        label = filename if len(pages) == 1 else f'{filename} (page {page_idx + 1})'

        # 1. greyscale
        grey_img = orig_img.convert('L')

        # 2. contrast enhancement
        contrast_img = ImageEnhance.Contrast(grey_img).enhance(contrast_factor)

        # 3. noise reduction
        blurred_img = contrast_img.filter(ImageFilter.GaussianBlur(radius=blur_radius))

        # 4. binarisation
        binary_img = blurred_img.point(lambda x: 0 if x < threshold else 255, '1')

        # display before and after
        plt.figure(figsize=(12, 6))
        plt.suptitle(f'Image {idx + 1}: {label}', fontsize=12)

        plt.subplot(1, 2, 1)
        plt.imshow(orig_img)
        plt.title('Original')
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(binary_img, cmap='gray')
        plt.title('Preprocessed')
        plt.axis('off')

        plt.show()

        # OCR with Tesseract
        text = pt.image_to_string(binary_img)
        ocr_results.append({'png_img': label, 'tesseract_ocr': text})

## Step 6: Save OCR output

This cell converts the `ocr_results` variable to a dataframe and saves it as a `.csv` file. This file is used as input in the next stage of the pipeline (`post_ocr_llm.ipynb`).

Update `output_path` to the location where you would like to save the file.

In [ ]:
df = pd.DataFrame(ocr_results)

# paste the full path where you would like to save the OCR output CSV
# example (Colab): "/content/drive/My Drive/your_project_folder/processed_imgs.csv"
# example (local): "/Users/yourname/Documents/THISTLE_project/processed_imgs.csv"
output_path = "insert_your_output_path/processed_imgs.csv"

df.to_csv(output_path, index=False)
print(f'Saved {len(df)} rows to {output_path}')